# Method 2 - k-means + PCA

Linear dimensionality reduction (PCA) followed by k-means. Serves as the linear
baseline against the SOM and the autoencoder methods.

**What changed vs the original**
- Shared `src/` preprocessing (identical pixels/bands to every other method).
- Bands are **not** standardised before PCA. Standardising is the usual advice,
  and it was tried - but `run_ablation_standardize.py` measures it as *harmful*
  here (Silhouette 0.310 -> 0.294, DB 0.991 -> 1.038), because all 219 bands are
  already the same physical unit (reflectance in [0,1]), so z-scoring only
  inflates noisy low-variance bands. The original choice was correct; the
  ablation is what settled it rather than an assertion either way.
- The label map gets the shared majority filter; intrinsic metrics are computed
  on the 10-D PCA scores; outputs saved via the shared `finalize_method`.

In [ ]:
# Make the shared src/ package importable regardless of CWD.
import sys, os
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.seeding import seed_everything
from src.preprocessing import load_and_preprocess_enmap
from src.results import finalize_method
from src.postprocess import majority_filter

seed_everything()          # seed numpy / random / torch from config.SEED
print(f"Seed = {config.SEED}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Raw reflectance (config.PCA_STANDARDIZE = False - see note above).
res = load_and_preprocess_enmap(config.ENMAP_TIF, standardize=config.PCA_STANDARDIZE)
X = res.data_valid

pca = PCA(n_components=config.PCA_N_COMPONENTS, random_state=config.SEED)
scores = pca.fit_transform(X)
cum = np.cumsum(pca.explained_variance_ratio_)
print(f"PCA {config.PCA_N_COMPONENTS} comps -> cumulative variance {cum[-1]*100:.2f}%")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(1, len(cum)+1), pca.explained_variance_ratio_*100, alpha=0.7, label="Individual")
ax2 = ax.twinx(); ax2.plot(range(1, len(cum)+1), cum*100, "o-", color="darkred", label="Cumulative")
ax.set_xlabel("Principal component"); ax.set_ylabel("Explained variance (%)")
ax2.set_ylabel("Cumulative (%)"); ax.set_title("PCA explained variance")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "pca_explained_variance.png", dpi=200); plt.show()

In [ ]:
# Elbow method for k.
inertias = []
Ks = range(2, 21)
for k in Ks:
    inertias.append(KMeans(n_clusters=k, random_state=config.SEED,
                           n_init=config.KMEANS_N_INIT).fit(scores).inertia_)
plt.figure(figsize=(9, 5))
plt.plot(list(Ks), inertias, "ko--", mfc="r"); plt.axvline(config.KMEANS_K, color="g", ls="--",
        label=f"k={config.KMEANS_K}")
plt.xlabel("k"); plt.ylabel("Inertia"); plt.title("Elbow method (PCA)"); plt.legend()
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "pca_elbow.png", dpi=200); plt.show()

In [ ]:
km = KMeans(n_clusters=config.KMEANS_K, random_state=config.SEED, n_init=config.KMEANS_N_INIT)
labels_raw = km.fit_predict(scores)

pca_map = finalize_method("pca", res, labels_raw, features_for_intrinsic=scores)

def plot_cluster_map(labels_spatial, extent, title):
    n = int(np.nanmax(labels_spatial)) + 1 if np.isfinite(np.nanmax(labels_spatial)) else 1
    fig, ax = plt.subplots(figsize=(12, 9))
    cmap = plt.get_cmap("rainbow", n).copy()
    cmap.set_bad(color="white")
    im = ax.imshow(labels_spatial, cmap=cmap, extent=extent, interpolation="nearest")
    cbar = plt.colorbar(im, ax=ax, label="Cluster", fraction=0.046)
    ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    return fig, ax

plot_cluster_map(pca_map, res.extent, f"k-means + PCA (k={config.KMEANS_K})")
plt.savefig(config.FIGURES_DIR / "pca_classification.png", dpi=200, bbox_inches="tight"); plt.show()